In [1]:
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error


from sklearn.pipeline import Pipeline #para encadear etapas (imputação, scaler, modelo etc.).
from sklearn.compose import ColumnTransformer # para aplicar transformações diferentes em colunas diferentes.
from sklearn.impute import SimpleImputer #para preencher valores faltantes.
from sklearn.preprocessing import OneHotEncoder, StandardScaler 
#para transformar variáveis categóricas em dummies.
#para padronizar dados numéricos (média 0, desvio 1).


# IMPORTANT: import from auto_sklearn2 (not autosklearn)
from auto_sklearn2 import AutoSklearnRegressor

In [2]:
penguins = sns.load_dataset('penguins')
print(penguins.shape)
penguins.head()

(344, 7)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


In [3]:
label = 'body_mass_g'

#penguins = penguins.dropna(subset = label).copy() #dropna remove NaN rowns
penguins = penguins.dropna().copy()
penguins.shape


(333, 7)

In [4]:
X, y = penguins.drop(label, axis = 1), penguins[label]
print(X.shape, y.shape)

(333, 6) (333,)


In [5]:
# 4) Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42,)

In [6]:
numeric_features = X.select_dtypes(include=np.number).columns
categorical_features = X.select_dtypes(exclude=np.number).columns
numeric_features, categorical_features

(Index(['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm'], dtype='object'),
 Index(['species', 'island', 'sex'], dtype='object'))

In [7]:
# Build preprocessing with IMPUTERS added
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    # ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
 #   ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop='first'))
])

preprocessor = ColumnTransformer([
   # ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [15]:
preprocessor.fit(X_train)
X_train_tr = preprocessor.transform(X_train)
X_test_tr = preprocessor.transform(X_test)

In [16]:
auto_sklearn = AutoSklearnRegressor(time_limit=120)
auto_sklearn.fit(X_train_tr,y_train)

,time_limit,120
,n_jobs,-1
,random_state,None
,scoring,'r2'


In [17]:
y_pred = auto_sklearn.predict(X_test_tr)

In [18]:
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Best Model: {auto_sklearn.best_params}")
print(f"R2 Score: {r2:.4f}")
print(f"Root Mean Squared Error:{rmse:.4f}")

Best Model: {'preprocessor': 'robust_scaler', 'regressor': 'ada_boost'}
R2 Score: 0.8692
Root Mean Squared Error:280.6130


In [19]:
perf = auto_sklearn.get_models_performance()
df = pd.DataFrame(list(perf.items()), columns = ['model', 'score'])
df.sort_values('score', ascending=False, inplace=True)
df.head(10)

,model,score
54,robust_scaler_ada_boost,0.840403
56,robust_scaler_bagging,0.837617
23,minmax_scaler_gradient_boosting,0.837505
15,standard_scaler_poisson,0.835478
33,minmax_scaler_extra_trees,0.835011
9,standard_scaler_decision_tree,0.834485
0,standard_scaler_random_forest,0.833435
48,robust_scaler_lasso,0.831337
34,minmax_scaler_bagging,0.831250
14,standard_scaler_huber,0.830798
